# Phase 2: Weather Features (Weeks 7-8)

Fetch historical weather data and compute month-specific climate features.

## Tasks
- Fetch city coordinates using Open-Meteo Geocoding API
- Query Open-Meteo Archive API for historical weather
- Compute monthly temperature and rainfall features
- Join weather with destination profiles from Phase 1
- Re-evaluate baselines with weather features

## Deliverable
✅ Comparison: Do weather features improve recommendation quality?

In [1]:
import sys
sys.path.insert(0, '../../')

import pandas as pd
import numpy as np
from pathlib import Path
import json
from config import *
from src.features.weather_features import WeatherFeatureBuilder

print("Phase 2: Weather Features (In Progress)")
print(f"Weather cache directory: {WEATHER_CACHE}")

Phase 2: Weather Features (In Progress)
Weather cache directory: C:\Users\Marlene\OneDrive\Desktop\[Tume] ML course\ML Project\data\cache\weather


## Step 1: Load Phase 1 Data

In [2]:
# Load businesses from major cities (output from Phase 1)
businesses_path = CACHE_DIR / "businesses_major_cities.parquet"

if not businesses_path.exists():
    print(f"⚠ {businesses_path} not found!")
    print("Please run Phase 1 notebooks first.")
else:
    businesses_major = pd.read_parquet(businesses_path)
    print(f"✓ Loaded {len(businesses_major)} businesses from major cities")
    print(f"  Cities: {businesses_major['city'].nunique()}")
    print(f"\nUnique cities:\n{sorted(businesses_major['city'].unique())}")

✓ Loaded 10341 businesses from major cities
  Cities: 39

Unique cities:
['Boise', 'Brandon', 'Brentwood', 'Carmel', 'Cherry Hill', 'Clearwater', 'Doylestown', 'Dunedin', 'Edmonton', 'Fishers', 'Franklin', 'Goleta', 'Greenwood', 'Hendersonville', 'Indianapolis', 'Kenner', 'King Of Prussia', 'Largo', 'Meridian', 'Metairie', 'Nashville', 'New Orleans', 'New Port Richey', 'Palm Harbor', 'Philadelphia', 'Pinellas Park', 'Reno', 'Riverview', 'Saint Louis', 'Saint Petersburg', 'Santa Barbara', 'Sparks', 'St. Louis', 'St. Petersburg', 'Tampa', 'Tucson', 'Wesley Chapel', 'West Chester', 'Wilmington']


## Step 2: Fetch Weather Data for Each City

In [3]:
# Initialize weather builder with cache
weather_builder = WeatherFeatureBuilder(cache_dir=WEATHER_CACHE)

# Extract unique cities
cities_df = pd.DataFrame({"city": businesses_major["city"].unique()})
print(f"Fetching weather for {len(cities_df)} cities...\n")

# Build weather features
weather_features = weather_builder.build_weather_features(cities_df)

if len(weather_features) > 0:
    print(f"\n✓ Successfully fetched weather for {len(weather_features)} city-months")
    print(f"\nWeather features shape: {weather_features.shape}")
    print(f"\nFirst few rows:\n{weather_features.head()}")
else:
    print("⚠ No weather features generated. Check API responses and logs above.")

Fetching weather for 39 cities...

Processing 1/39: St. Petersburg
  ✓ Got 12 months of data
Processing 2/39: Nashville
  ✓ Got 12 months of data
Processing 3/39: Indianapolis
  ✓ Got 12 months of data
Processing 4/39: Philadelphia
  ✓ Got 12 months of data
Processing 5/39: Santa Barbara
  ✓ Got 12 months of data
Processing 6/39: Tucson
  ✓ Got 12 months of data
Processing 7/39: New Orleans
  ✓ Got 12 months of data
Processing 8/39: Clearwater
  ✓ Got 12 months of data
Processing 9/39: Largo
  ✓ Got 12 months of data
Processing 10/39: Tampa
  ✓ Got 12 months of data
Processing 11/39: Saint Petersburg
  ✓ Got 12 months of data
Processing 12/39: Brandon
  ✓ Got 12 months of data
Processing 13/39: Kenner
  ✓ Got 12 months of data
Processing 14/39: Boise
  ✓ Got 12 months of data
Processing 15/39: Sparks
  ✓ Got 12 months of data
Processing 16/39: King Of Prussia
  ✓ Got 12 months of data
Processing 17/39: New Port Richey
  ✓ Got 12 months of data
Processing 18/39: Hendersonville
  ✓ Got 1

## Step 3: Save Weather Features

In [4]:
if len(weather_features) > 0:
    output_path = CACHE_DIR / "weather_features_by_city_month.parquet"
    weather_features.to_parquet(output_path)
    print(f"✓ Saved weather features to {output_path}")
    
    # Also save summary stats by city (aggregated across months)
    weather_summary = weather_features.groupby("city").agg({
        "avg_temp": ["mean", "min", "max", "std"],
        "total_precipitation": ["mean", "min", "max"],
        "lat": "first",
        "lon": "first"
    }).reset_index()
    weather_summary.columns = ["_".join(col).strip("_") for col in weather_summary.columns]
    weather_summary.rename(columns={"city_": "city"}, inplace=True)
    
    summary_path = CACHE_DIR / "weather_summary_by_city.parquet"
    weather_summary.to_parquet(summary_path)
    print(f"✓ Saved weather summary to {summary_path}")
    print(f"\nWeather summary by city:\n{weather_summary.head(10)}")

✓ Saved weather features to C:\Users\Marlene\OneDrive\Desktop\[Tume] ML course\ML Project\data\cache\weather_features_by_city_month.parquet
✓ Saved weather summary to C:\Users\Marlene\OneDrive\Desktop\[Tume] ML course\ML Project\data\cache\weather_summary_by_city.parquet

Weather summary by city:
          city  avg_temp_mean  avg_temp_min  avg_temp_max  avg_temp_std  \
0        Boise      11.914454      1.064516     26.783871      9.623664   
1      Brandon      23.705592     17.432258     29.253226      4.032792   
2    Brentwood      10.494927     -1.069643     23.083871      8.509901   
3       Carmel      12.724110      2.153226     23.970968      8.122812   
4  Cherry Hill      14.397966      5.208929     26.479032      7.971588   
5   Clearwater      23.477429     17.093548     29.090323      4.106175   
6   Doylestown      13.124272      4.028571     24.988710      7.864960   
7      Dunedin      11.155063      6.595161     16.108929      3.198170   
8     Edmonton       5.4314

## Step 4: Merge Weather with Destination Profiles

In [5]:
# Load destination profiles from Phase 1 (if available)
dest_profiles_path = CACHE_DIR / "destination_profiles.parquet"

if dest_profiles_path.exists():
    dest_profiles = pd.read_parquet(dest_profiles_path)
    print(f"✓ Loaded destination profiles: {dest_profiles.shape}")
    print(f"\nColumns: {list(dest_profiles.columns)}")
    
    # If profiles have monthly data already, merge weather
    if "month" in dest_profiles.columns:
        dest_with_weather = dest_profiles.merge(
            weather_features[["city", "month", "avg_temp", "avg_precipitation", "temp_std"]],
            on=["city", "month"],
            how="left"
        )
        print(f"\n✓ Merged destination profiles with weather")
        print(f"  New shape: {dest_with_weather.shape}")
    else:
        print("\nNote: Destination profiles don't have monthly structure.")
        print("Weather features are city-month level; will need to handle this in Phase 4.")
else:
    print(f"⚠ Destination profiles not found at {dest_profiles_path}")
    print("  Will proceed with weather features alone for Phase 2 analysis.")

⚠ Destination profiles not found at C:\Users\Marlene\OneDrive\Desktop\[Tume] ML course\ML Project\data\cache\destination_profiles.parquet
  Will proceed with weather features alone for Phase 2 analysis.


## Step 5: Weather Feature Statistics

In [6]:
if len(weather_features) > 0:
    print("Weather Feature Statistics (across all cities and months):\n")
    print(weather_features[[
        "avg_temp", "avg_temp_max", "avg_temp_min",
        "total_precipitation", "avg_precipitation", "temp_std"
    ]].describe())
    
    print("\n" + "="*70)
    print("Weather by Season (Northern Hemisphere):\n")
    
    season_map = {1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                  6: "Summer", 7: "Summer", 8: "Summer",
                  9: "Fall", 10: "Fall", 11: "Fall", 12: "Winter"}
    weather_features["season"] = weather_features["month"].map(season_map)
    
    season_stats = weather_features.groupby("season").agg({
        "avg_temp": ["mean", "min", "max"],
        "total_precipitation": ["mean"],
    }).round(2)
    print(season_stats)

Weather Feature Statistics (across all cities and months):

         avg_temp  avg_temp_max  avg_temp_min  total_precipitation  \
count  468.000000    468.000000    468.000000           468.000000   
mean    17.122317     21.947254     12.297380            85.859615   
std      8.477059      8.710566      8.379964            56.669019   
min     -9.108929     -5.125000    -13.092857             0.000000   
25%     11.296429     15.970000      6.338047            39.775000   
50%     17.756425     23.061290     13.647976            76.900000   
75%     24.076667     29.048387     18.781532           126.125000   
max     34.319355     41.500000     27.735484           345.400000   

       avg_precipitation    temp_std  
count         468.000000  468.000000  
mean            2.816180    3.000452  
std             1.844974    1.294135  
min             0.000000    0.514149  
25%             1.283065    1.972306  
50%             2.535484    2.978807  
75%             4.102419    3.826396

## Step 6: Temperature and Precipitation Ranges by City

In [7]:
if len(weather_features) > 0:
    city_weather = weather_features.groupby("city").agg({
        "avg_temp": ["min", "mean", "max"],
        "total_precipitation": ["sum", "mean"],
    }).round(2)
    city_weather.columns = ["_".join(col) for col in city_weather.columns]
    city_weather = city_weather.sort_values(("avg_temp_mean"), ascending=False)
    print("Temperature and Precipitation by City (annual summary):\n")
    print(city_weather)

Temperature and Precipitation by City (annual summary):

                  avg_temp_min  avg_temp_mean  avg_temp_max  \
city                                                          
Tampa                    17.31          23.98         29.90   
Pinellas Park            17.54          23.90         29.57   
St. Petersburg           17.34          23.90         29.64   
Riverview                17.62          23.79         29.30   
Brandon                  17.43          23.71         29.25   
Clearwater               17.09          23.48         29.09   
Largo                    17.09          23.48         29.09   
Wesley Chapel            17.06          23.40         29.04   
Palm Harbor              16.88          23.38         28.93   
New Port Richey          16.68          22.98         28.55   
Kenner                   14.05          22.52         31.80   
New Orleans              14.07          22.49         31.90   
Metairie                 14.07          22.49         31.90  

## Step 7: Prepare Data for Phase 4 (Models)

Create features ready to use in ML models. This bridges Phase 2 → Phase 4.

In [8]:
if len(weather_features) > 0:
    # Create pivot tables for easier feature engineering
    # This converts city-month data to city-level features
    
    weather_wide = weather_features.pivot_table(
        index="city",
        columns="month",
        values="avg_temp",
        aggfunc="first"
    )
    weather_wide.columns = [f"temp_month_{m}" for m in weather_wide.columns]
    
    precip_wide = weather_features.pivot_table(
        index="city",
        columns="month",
        values="total_precipitation",
        aggfunc="first"
    )
    precip_wide.columns = [f"precip_month_{m}" for m in precip_wide.columns]
    
    # Combine with summary stats
    weather_for_models = weather_features.drop_duplicates("city")[["city", "lat", "lon"]].set_index("city")
    weather_for_models = weather_for_models.join(weather_wide)
    weather_for_models = weather_for_models.join(precip_wide)
    weather_for_models = weather_for_models.reset_index()
    
    # Add aggregate features
    weather_for_models["annual_avg_temp"] = weather_features.groupby("city")["avg_temp"].mean().values
    weather_for_models["annual_precipitation"] = weather_features.groupby("city")["total_precipitation"].sum().values
    weather_for_models["temp_range"] = (
        weather_features.groupby("city")["avg_temp"].max().values -
        weather_features.groupby("city")["avg_temp"].min().values
    )
    
    # Save for Phase 4
    model_features_path = CACHE_DIR / "weather_features_for_models.parquet"
    weather_for_models.to_parquet(model_features_path)
    print(f"✓ Saved model-ready weather features to {model_features_path}")
    print(f"\nShape: {weather_for_models.shape}")
    print(f"\nFirst few rows:\n{weather_for_models.iloc[:, :8].head()}")

✓ Saved model-ready weather features to C:\Users\Marlene\OneDrive\Desktop\[Tume] ML course\ML Project\data\cache\weather_features_for_models.parquet

Shape: (39, 30)

First few rows:
             city       lat        lon  temp_month_1  temp_month_2  \
0  St. Petersburg  27.77086  -82.67927     17.343548     20.214286   
1       Nashville  36.16589  -86.78444      7.374194      9.821429   
2    Indianapolis  39.76838  -86.15804      2.469355      4.423214   
3    Philadelphia  39.95238  -75.16362      5.616129      5.137500   
4   Santa Barbara  34.42083 -119.69819     11.603226     11.166071   

   temp_month_3  temp_month_4  temp_month_5  
0     21.732258     24.263333     25.667742  
1     10.346774     14.808333     20.393548  
2      5.412903     11.348333     17.404839  
3      6.512903     14.538333     16.630645  
4     11.693548     15.535000     15.877419  


## Summary

Phase 2 complete! Weather features have been:
- ✅ Fetched from Open-Meteo API for all major cities
- ✅ Processed into monthly and annual features
- ✅ Cached for future use
- ✅ Formatted for Phase 4 model training

Next: Phase 3 (Review Embeddings) or Phase 4 (ML Models with weather features)